# MarketLens Stage 7: LLM Market Summary Test

This notebook tests an LLM-powered summary feature.

Input:
- `data/processed/marketlens_features.csv`

Output:
- Plain-English market summary generated only from calculated metrics.

Rule:
The LLM must not invent data, forecasts, or trading recommendations.

#### Imports and project root

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"C:\Users\Wilson\Documents\OpenAI\MarketLens")
os.chdir(PROJECT_ROOT)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()

print("Current working directory:")
print(Path.cwd())

Current working directory:
C:\Users\Wilson\Documents\OpenAI\MarketLens


#### Load feature dataset

In [3]:
features_path = PROCESSED_DIR / "marketlens_features.csv"

if not features_path.exists():
    raise FileNotFoundError(f"Missing file: {features_path}")

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.tail())

Dataset shape: (2136, 17)
Columns:
['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dgs10', 'daily_return', 'cumulative_return', 'moving_average_20', 'moving_average_50', 'rolling_volatility_20', 'rsi_14', 'running_max_close', 'drawdown', 'trend_status']


,date,open,high,low,close,adj_close,volume,dgs10,daily_return,cumulative_return,moving_average_20,moving_average_50,rolling_volatility_20,rsi_14,running_max_close,drawdown,trend_status
2131,2026-06-26,728.950012,736.530029,716.580017,728.989990,728.989990,71034000,4.38,-0.007231,1.712319,743.601999,734.351400,0.167221,44.927706,759.570007,-0.040260,Below 50-day average
2132,2026-06-29,736.530029,741.559998,732.090027,741.000000,741.000000,58035200,4.38,0.016475,1.757004,742.828000,735.138201,0.178805,50.939627,759.570007,-0.024448,Above 50-day average
2133,2026-06-30,741.289978,748.020020,740.890015,746.770020,746.770020,55626000,4.44,0.007787,1.778472,742.239502,735.870801,0.181082,54.943059,759.570007,-0.016852,Above 50-day average
2134,2026-07-01,745.000000,749.440002,742.380005,745.760010,745.760010,47100900,4.48,-0.001353,1.774715,741.549002,736.611602,0.180925,61.589335,759.570007,-0.018181,Above 50-day average
2135,2026-07-02,747.400024,751.309998,740.030029,744.780029,744.780029,57447800,4.48,-0.001314,1.771068,741.076004,737.425602,0.179475,54.596661,759.570007,-0.019472,Above 50-day average


#### Build latest metrics dictionary

In [5]:
df = df.dropna(subset=["date", "close"]).sort_values("date").reset_index(drop=True)

latest = df.iloc[-1]

metrics = {
    "latest_date": str(latest["date"].date()),
    "latest_close": round(float(latest["close"]), 2),
    "latest_daily_return_percent": round(float(latest["daily_return"]) * 100, 2),
    "latest_cumulative_return_percent": round(float(latest["cumulative_return"]) * 100, 2),
    "latest_rolling_volatility_20_percent": round(float(latest["rolling_volatility_20"]) * 100, 2),
    "latest_rsi_14": round(float(latest["rsi_14"]), 2),
    "latest_drawdown_percent": round(float(latest["drawdown"]) * 100, 2),
    "latest_dgs10": round(float(latest["dgs10"]), 2),
    "latest_trend_status": str(latest["trend_status"]),
}

metrics

{'latest_date': '2026-07-02',
 'latest_close': 744.78,
 'latest_daily_return_percent': -0.13,
 'latest_cumulative_return_percent': 177.11,
 'latest_rolling_volatility_20_percent': 17.95,
 'latest_rsi_14': 54.6,
 'latest_drawdown_percent': -1.95,
 'latest_dgs10': 4.48,
 'latest_trend_status': 'Above 50-day average'}

#### Create controlled LLM prompt

In [6]:
def build_market_summary_prompt(metrics: dict) -> str:
    return f"""
You are generating a portfolio-project dashboard summary.

Use only the metrics provided below.
Do not invent causes, forecasts, recommendations, or trading advice.
Do not claim that the data proves profitability.
Use clear plain English.

Metrics:
- Latest date: {metrics["latest_date"]}
- Latest SPY close: {metrics["latest_close"]}
- Latest daily return: {metrics["latest_daily_return_percent"]}%
- Cumulative return from dataset start: {metrics["latest_cumulative_return_percent"]}%
- 20-day annualized rolling volatility: {metrics["latest_rolling_volatility_20_percent"]}%
- RSI 14: {metrics["latest_rsi_14"]}
- Current drawdown: {metrics["latest_drawdown_percent"]}%
- DGS10 10-Year Treasury yield: {metrics["latest_dgs10"]}
- Trend status: {metrics["latest_trend_status"]}

Write 4 short bullet points:
1. Price and trend
2. Return and drawdown
3. Volatility and RSI
4. Macro note from DGS10

End with this exact sentence:
This summary is descriptive only and is not investment advice.
""".strip()


prompt = build_market_summary_prompt(metrics)

print(prompt)

You are generating a portfolio-project dashboard summary.

Use only the metrics provided below.
Do not invent causes, forecasts, recommendations, or trading advice.
Do not claim that the data proves profitability.
Use clear plain English.

Metrics:
- Latest date: 2026-07-02
- Latest SPY close: 744.78
- Latest daily return: -0.13%
- Cumulative return from dataset start: 177.11%
- 20-day annualized rolling volatility: 17.95%
- RSI 14: 54.6
- Current drawdown: -1.95%
- DGS10 10-Year Treasury yield: 4.48
- Trend status: Above 50-day average

Write 4 short bullet points:
1. Price and trend
2. Return and drawdown
3. Volatility and RSI
4. Macro note from DGS10

End with this exact sentence:
This summary is descriptive only and is not investment advice.


#### Test OpenAI API call

In [ ]:
from openai import OpenAI, RateLimitError, APIError, AuthenticationError

api_key = os.getenv("OPENAI_API_KEY")
model = os.getenv("OPENAI_MODEL", "gpt-5.5")

def generate_fallback_summary(metrics: dict) -> str:
    return f"""
- Price and trend: On {metrics["latest_date"]}, SPY closed at {metrics["latest_close"]}. The trend status is {metrics["latest_trend_status"]}.
- Return and drawdown: The latest daily return is {metrics["latest_daily_return_percent"]}%. The cumulative return from the dataset start is {metrics["latest_cumulative_return_percent"]}%, and the current drawdown is {metrics["latest_drawdown_percent"]}%.
- Volatility and RSI: The 20-day annualized rolling volatility is {metrics["latest_rolling_volatility_20_percent"]}%. The RSI 14 value is {metrics["latest_rsi_14"]}.
- Macro note from DGS10: The DGS10 10-Year Treasury yield value in the dataset is {metrics["latest_dgs10"]}.

This summary is descriptive only and is not investment advice.
""".strip()


summary_mode = "fallback"

if not api_key or api_key == "PASTE_YOUR_OPENAI_KEY_HERE":
    summary_text = generate_fallback_summary(metrics)
    print("OPENAI_API_KEY missing. Using fallback summary.\n")
else:
    try:
        client = OpenAI(api_key=api_key)

        response = client.responses.create(
            model=model,
            input=prompt,
        )

        summary_text = response.output_text
        summary_mode = "openai_api"

    except RateLimitError as error:
        summary_text = generate_fallback_summary(metrics)
        summary_mode = "fallback_insufficient_quota"
        print("OpenAI quota error. Using fallback summary.\n")
        print(error)

    except AuthenticationError as error:
        summary_text = generate_fallback_summary(metrics)
        summary_mode = "fallback_authentication_error"
        print("OpenAI authentication error. Using fallback summary.\n")
        print(error)

    except APIError as error:
        summary_text = generate_fallback_summary(metrics)
        summary_mode = "fallback_api_error"
        print("OpenAI API error. Using fallback summary.\n")
        print(error)

print("\nSummary mode:", summary_mode)
print("\nGenerated summary:\n")
print(summary_text)

#### Save LLM summary report

In [ ]:
from datetime import datetime

report_path = REPORTS_DIR / "llm_summary_check.txt"

report = f"""
MarketLens LLM Summary Check

Status: LLM summary workflow tested.
Timestamp: {datetime.now()}

Input file:
- data/processed/marketlens_features.csv

Configured model:
- {model}

Summary mode:
- {summary_mode}

Metrics passed to summary workflow:
{metrics}

Generated summary:
{summary_text}

Result:
The live OpenAI API call could not be completed because the account returned insufficient_quota.
The fallback summary mode worked and used only calculated dashboard metrics.

Rule:
The summary used only calculated dashboard metrics and did not provide trading advice.
"""

report_path.write_text(report.strip(), encoding="utf-8")

print(f"Report saved to: {report_path}")